# 10_Explainability.ipynb
======================
This notebook demonstrates explainability utilities for the customer support ticket classification model using LIME (Local Interpretable Model-agnostic Explanations).

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if REPO_ROOT.name != "SupportAI" and (REPO_ROOT).exists():
    REPO_ROOT = REPO_ROOT
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from pathlib import Path

from src.evaluation.explainability import TicketExplainer
from src.utils.constants import OUTPUT_DIR

## 1. Initialise the TicketExplainer
We load the fine-tuned sequence classifier from the `outputs/models/best_model` directory.

In [2]:
model_dir = OUTPUT_DIR / "models" / "best_model"
explainer = TicketExplainer(model_dir=model_dir)

[07/13/26 20:47:08] INFO     Loading model and tokenizer for explainability from                                   
                             C:\Users\gunav\Downloads\SupportAI\outputs\models\best_model

## 2. Generate Prediction Explanation
We choose a ticket text representing a customer issue and generate word attributions.

In [3]:
ticket_text = "My computer laptop has a black screen and won't turn on. Please help!"
result = explainer.explain_ticket(ticket_text, num_features=8, num_samples=100)

print(f"Predicted Class: {result['predicted_class']}")
print(f"Confidence: {result['predicted_probability']:.2%}")

Predicted Class: passcode_forgotten
Confidence: 18.75%


## 3. Visualize Attributions
We print the list of top words contributing to the classification decision.

In [4]:
explainer.visualize_explanation(result)

Prediction Explanation Summary

Predicted Class: passcode_forgotten

Confidence Score: 18.75%

    Word Attributions for Class 'passcode_forgotten'     
┏━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ Rank ┃ Word / Feature ┃ Attribution Score ┃ Direction ┃
┡━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│  1   │ screen         │           +0.0929 │ Positive  │
│  2   │ turn           │           -0.0854 │ Negative  │
│  3   │ t              │           +0.0720 │ Positive  │
│  4   │ won            │           +0.0704 │ Positive  │
│  5   │ has            │           -0.0470 │ Negative  │
│  6   │ laptop         │           -0.0289 │ Negative  │
│  7   │ black          │           +0.0186 │ Positive  │
│  8   │ on             │           +0.0117 │ Positive  │
└──────┴────────────────┴───────────────────┴───────────┘

## 4. Export LIME HTML Explanation
We save the local explanation report as an HTML file for interactive visualization.

In [5]:
output_html_path = OUTPUT_DIR / "metrics" / "plots" / "lime_explanation.html"
output_html_path.parent.mkdir(parents=True, exist_ok=True)
output_html_path.write_text(result["explanation_html"], encoding="utf-8")
print(f"Saved LIME HTML explanation report to: {output_html_path}")

Saved LIME HTML explanation report to: C:\Users\gunav\Downloads\SupportAI\outputs\metrics\plots\lime_explanation.html


In [6]:
# Export Phase Manifest
from src.api.app import get_git_commit
from src.utils.artifacts import save_yaml

manifest = {
    "phase": "10_Explainability",
    "explainability_check": "LIME report generated successfully",
    "git_commit": get_git_commit(),
}
save_yaml(manifest, REPO_ROOT / "outputs" / "manifests" / "phase_10_explainability.yaml")
print("YAML manifest saved successfully:")
print(manifest)


[07/13/26 20:47:10] INFO     Loading faiss with AVX2 support.

                    INFO     Could not load library with AVX2 support due to:                                      
                             ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")

                    INFO     Loading faiss.

                    INFO     Successfully loaded faiss.

[07/13/26 20:47:12] INFO     Saving YAML artifact to:                                                              
                             C:\Users\gunav\Downloads\SupportAI\outputs\manifests\phase_10_explainability.yaml

YAML manifest saved successfully:
{'phase': '10_Explainability', 'explainability_check': 'LIME report generated successfully', 'git_commit': 'ef9a0498221c5c43373fcf9e951987614174868f'}
